# UMUD R76T — A76 rozcięgna + F76 pęczki (Colab, GPU)

**Cel:** poprawić pomiar na obrazie, zachowując sprawdzoną regułę FL: 5 najmniej ekstrapolowanych pęczków, próg 0,6.
Notebook przygotowany po analizie dziennika z 26.10.2026, źródeł R45T/R71T i aktualnych artefaktów na Drive.
Punkt odniesienia z dziennika: **B6 = 0,20823**; nowy R76 **nie ma jeszcze wyniku na serwerze**.

1. **A76:** ConvNeXt-Tiny U-Net, 1024 px, maska rozcięgien + pomocnicze cele ich wewnętrznych krawędzi.
   Strata obejmuje położenie i nachylenie krawędzi. Maska zachowuje BCE z `pos_weight=1`.
2. **F76:** ResNet34 U-Net, 1024 px, rurki wokół linii ekspertów + kierunek osiowy. Inny enkoder niż R22/F71;
   wykorzystujemy ten sam kontrakt etykiet i tę samą regułę wyboru fragmentów.
3. **Rzetelniejsza walidacja:** grupy sekwencji, scalanie identycznych / odbitych obrazów i bardzo podobnych klatek.
   Nowe modele startują od ImageNet, więc nowe foldy nie dziedziczą przecieku ze starych wag UMUD.
4. **EMA, TTA, wznawianie w środku epoki:** checkpoint obejmuje model, EMA, optymalizator, skaler, pozycję w epoce i RNG.

**Uruchomienie:** w Colab wybierz GPU (L4/A100, ewentualnie T4), potem **Uruchom wszystko** i zatwierdź dostęp do własnego Drive.
Domyślnie najpierw kończy się 5 foldów A76, potem 5 foldów F76. Limit sesji 10,5 h; pełny trening może wymagać kolejnej sesji.
Po rozłączeniu uruchom ten sam plik z tym samym `RUN_NAME`. Czas docelowego treningu nie był tu mierzony na GPU;
postęp i czas epok będą widoczne podczas biegu.

Wyniki trafią do `UMUD_Challenge/outputs/R76T_v1_0_0/`. Eksporty po 1, 3 i 5 foldach dają wcześniejszy materiał do porównań.
To notebook treningu i eksportu cech; końcowy tor B6/R75 wymaga potem podłączenia wybranego wariantu i pełnej inferencji od obrazów.


In [8]:
# 1. USTAWIENIA — na zwykłym biegu zmieniaj tylko tę komórkę.
RUN_NAME = 'R76T_v1_0_0'    # ten sam katalog = wznowienie; inny eksperyment = nowa nazwa
TRAIN_A76 = True
TRAIN_F76 = True            # rozpocznie się po ukończeniu A76
RUN_EXPORT = True
FOLDS_A76 = [0, 1, 2, 3, 4]
FOLDS_F76 = [0, 1, 2, 3, 4]
EXPORT_AFTER = [1, 3, 5]
SESSION_HOURS = 10.5        # zachowuje czas na zapis przed typowym rozłączeniem Colaba
EPOCHS_A76 = 24
EPOCHS_F76 = 20
CHECK_IMAGE_SHA = True
RUN_SELF_TEST = True


In [9]:
# 2. ŚRODOWISKO I SAMOWYSTARCZALNY KOD (GPL-3.0-or-later).
# Pełne pliki .py zostaną wypakowane i zapisane w katalogu code/ na Twoim Drive.
import os,sys,time,json,hashlib,base64,zlib,subprocess,importlib.util,shutil,zipfile
from pathlib import Path
T_START=time.time()
FIXTURE=os.environ.get('R76T_FIXTURE_ROOT')  # tylko automatyczne testy lokalne autora
if FIXTURE:
    DRIVE_ROOT=Path(FIXTURE)/'MyDrive';LOCAL=Path(FIXTURE)/'content';LOCAL.mkdir(parents=True,exist_ok=True)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT=Path('/content/drive/MyDrive');LOCAL=Path('/content')
PROJECT=DRIVE_ROOT/'UMUD_Challenge';RUN=PROJECT/'outputs'/RUN_NAME;RUN.mkdir(parents=True,exist_ok=True)
PACKAGE_SHA='af5a99e29ddb171a05276708a81eca586745e7cc4a2eadd35348a01cd475a1d8'
PAYLOAD='...'
raw=zlib.decompress(base64.b64decode(PAYLOAD))
if hashlib.sha256(raw).hexdigest()!=PACKAGE_SHA:raise RuntimeError('Uszkodzony kod notebooka — wczytaj oryginalny plik ponownie')
SRC=LOCAL/('r76t_src_'+PACKAGE_SHA[:12]);SRC.mkdir(parents=True,exist_ok=True)
for rel,content in json.loads(raw).items():
    path=SRC/rel
    if not path.resolve().is_relative_to(SRC.resolve()):raise ValueError('Invalid source path')
    path.parent.mkdir(parents=True,exist_ok=True);path.write_text(content)
for module,package in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('cv2','opencv-python-headless'),('skimage','scikit-image'),('timm','timm')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
if importlib.util.find_spec('torch') is None or importlib.util.find_spec('torchvision') is None:
    raise RuntimeError('Potrzebne zgodne torch + torchvision. Wybierz standardowe środowisko GPU Colab.')
sys.path.insert(0,str(SRC))
_names=['r45_core','r45_models','r45_decode','r45_apo','r45_infer','r71_geom','r71_train','r76_data','r76_train','r76_export','r76_selftest']
for name in _names:sys.modules.pop(name,None)
import numpy as np,pandas as pd,torch,torchvision,cv2
import r45_core,r45_infer,r76_data,r76_train,r76_export,r76_selftest
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE=='cpu' and not FIXTURE:raise RuntimeError('Brak GPU. Colab: Środowisko wykonawcze → Zmień typ → GPU.')
if DEVICE=='cuda':
    torch.backends.cudnn.benchmark=True
    torch.backends.cuda.matmul.allow_tf32=True
    torch.set_float32_matmul_precision('high')
GPU=torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU (testy)'
def log(*args):
    msg=' '.join(str(a) for a in args);print(msg,flush=True)
    with (RUN/'LOG.txt').open('a') as f:f.write(time.strftime('%Y-%m-%d %H:%M:%S ')+msg+'\n')
def show(frame,n=30):print(frame.head(n).to_string(index=False),flush=True)
code_dir=RUN/'code';code_dir.mkdir(exist_ok=True)
archive=code_dir/f'R76T_source_{PACKAGE_SHA[:12]}.zip'
if not archive.exists():
    tmp=archive.with_suffix('.tmp.zip')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for rel,content in json.loads(raw).items():z.writestr(rel,content)
    tmp.replace(archive)
r45_core.atomic_json(code_dir/'PACKAGE.json',{'source_sha256':PACKAGE_SHA,'archive':archive.name})
r45_core.atomic_json(RUN/'ENVIRONMENT.json',{'python':sys.version,'torch':torch.__version__,'torchvision':torchvision.__version__,
    'numpy':np.__version__,'pandas':pd.__version__,'opencv':cv2.__version__,'gpu':GPU,'cuda':torch.version.cuda})
CFG_A=dict(r76_train.APO_CONFIG,epochs=EPOCHS_A76)
CFG_F=dict(r76_train.FASC_CONFIG,epochs=EPOCHS_F76)
if FIXTURE:
    override=json.loads(os.environ.get('R76T_TEST_OVERRIDES','{}'));CFG_A.update(override);CFG_F.update(override)
OUT_A=RUN/'A76';OUT_F=RUN/'F76';OUT_A.mkdir(exist_ok=True);OUT_F.mkdir(exist_ok=True)
ASSETS=SRC/'assets';TRAIN_ROOT=PROJECT/'data'/'competition_train_v1'
DEADLINE=T_START+SESSION_HOURS*3600
BUDGET_HIT=False
log('R76T |',GPU,'| kod',PACKAGE_SHA[:12],'| AMP',str(r76_train.precision_mode(DEVICE)[0]),'|',RUN)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
R76T | NVIDIA L4 | kod af5a99e29ddb | AMP torch.bfloat16 | /content/drive/MyDrive/UMUD_Challenge/outputs/R76T_v1_0_0


In [10]:
# 3. KONTROLE POCZĄTKOWE I ZAMROŻONE ŹRÓDŁA PORÓWNAWCZE.
if RUN_SELF_TEST:
    r45_core.atomic_json(RUN/'SELF_TEST.json',r76_selftest.run(log))
REGISTRY=r76_export.baseline_registry(PROJECT,DRIVE_ROOT)
r45_core.atomic_json(RUN/'BASELINE_REGISTRY.json',REGISTRY)
test=r45_infer.load_manifest(ASSETS/'test_manifest.csv',PROJECT,CHECK_IMAGE_SHA)
osf=r45_infer.load_manifest(ASSETS/'osf_manifest.csv',PROJECT,CHECK_IMAGE_SHA)
MANIFESTS={'test':test,'osf':osf}
log('Potwierdzone: R22 5/5, v2 5/5, F71 5/5 i R12B; obrazy testu:',len(test),'OSF:',len(osf))
log('Dotychczasowe modele są tylko źródłem porównania — nowe modele startują od ImageNet.')
log('Kąty w eksportach są w układzie pikseli; dotychczasowa korekta proporcji źródła pozostaje w torze B6/R75.')


PASS: subpixel edge coordinates, legacy decoder equality, anatomical guard, fixed lex5
PASS: apo network forward/backward
PASS: fasc network forward/backward
Potwierdzone: R22 5/5, v2 5/5, F71 5/5 i R12B; obrazy testu: 309 OSF: 35
Dotychczasowe modele są tylko źródłem porównania — nowe modele startują od ImageNet.
Kąty w eksportach są w układzie pikseli; dotychczasowa korekta proporcji źródła pozostaje w torze B6/R75.


## Dlaczego ta kolejność

W aktualnym OOF v2 zapisano 76 niezgodnych par rozcięgien na 1048 obrazach, oprócz brakujących pomiarów.
Sama wysoka mediana Dice (~0,87) tego nie pokazuje. A76 oceniamy więc również przez liczbę błędów pary,
MAE i 90. percentyl błędu MT oraz kąt głębokiego rozcięgna. Raport zawiera błędy wszystkich wykrytych par
i osobno zgodnych par — błędne pary nie znikają z podsumowania.

Nowe foldy mogą być trudniejsze od historycznych: łączą duplikaty między starymi grupami. Nie jest to
identyczny eksperyment OOF z v2; historycznych median nie należy odejmować jako czystego przyrostu jakości.
Grupy po nazwach i podobieństwie obrazu są przybliżeniem sekwencji, nie potwierdzonym identyfikatorem pacjenta.

Kontrola OSF jest pomocnicza. Z dziennika wiemy, że OSF i symulator potrafiły wskazać złą regułę FL.
Dlatego nie zmieniamy miękkich rang, liczby pęczków ani smoothingu ich kątów na podstawie tego proxy.


In [11]:
# 4. FUNKCJE TRENINGU I EKSPORTU. Eksport jest wznawialny po każdym obrazie.
def export_now():
    return r76_export.export_stage(RUN,PROJECT,REGISTRY,MANIFESTS,ASSETS,{'apo':OUT_A,'fasc':OUT_F},
        LOCAL/'r76_baseline_maps',PACKAGE_SHA,DEVICE,deadline=DEADLINE,log=log)

def run_task(task,frame,cache,pretrained,cfg,folds,out):
    global BUDGET_HIT
    for fold in folds:
        if time.time()>=DEADLINE:
            BUDGET_HIT=True;break
        try:
            result=r76_train.train_fold(frame,cache,out,pretrained,cfg,fold,task,DEVICE,deadline=DEADLINE,log=log)
            log(task,'fold',fold,'gotowy:',json.dumps(result['oof_summary']))
            complete=r76_train.complete_folds(out)
            if RUN_EXPORT and len(complete) in EXPORT_AFTER:export_now()
        except r76_train.SessionBudgetReached as e:
            BUDGET_HIT=True;log(str(e),'— uruchom wszystko ponownie w nowej sesji z tym samym RUN_NAME');break
    reports=[]
    for path in sorted(out.glob('fold_*/DONE.json')):
        d=json.loads(path.read_text());reports.append({'fold':d['fold'],**d['oof_summary']})
    if reports:
        table=pd.DataFrame(reports);table.to_csv(out/'OOF_SUMMARY.csv',index=False);show(table)
    log(task,'ukończone foldy:',r76_train.complete_folds(out))


In [12]:
# 5. A76 — ROZCIĘGNA. Dane i cache są odtwarzalne z sum źródeł.
if TRAIN_A76 and not BUDGET_HIT:
    paired_a=r76_data.pairs(TRAIN_ROOT/'apo_images',TRAIN_ROOT/'apo_masks')
    frame_a,audit_a=r76_data.audit_and_folds(paired_a,OUT_A,n_folds=5,seed=CFG_A['seed'],
        legacy_folds=PROJECT/'outputs'/'R45T_v1_0_0'/'apo2'/'FOLDS.csv',log=log)
    if len(frame_a)!=1048 and not FIXTURE:log('UWAGA: liczba par różni się od historycznych 1048:',len(frame_a))
    cache_a=LOCAL/f'r76_apo_cache_{CFG_A["side"]}_{audit_a["identity"][:10]}'
    r76_data.build_cache(frame_a,cache_a,OUT_A/f'cache_{CFG_A["side"]}.tar','apo',CFG_A,log)
    pretrained_a=r76_train.prepare_pretrained(PROJECT,RUN,'apo')
    run_task('apo',frame_a,cache_a,pretrained_a,CFG_A,FOLDS_A76,OUT_A)
else:log('A76: pominięto trening w tej sesji')


audit: 200/1048
audit: 400/1048
audit: 600/1048
audit: 800/1048
audit: 1000/1048
split: {'identity': '5c72588319d8c667831dd6e042acebc9a4f60f4628d301fcb0016b9bb00e35cd', 'samples': 1048, 'groups': 100, 'largest_group': 60, 'exact_or_mirrored_duplicates': 20, 'near_duplicate_links': 21, 'folds': {'0': 208, '1': 210, '2': 210, '3': 210, '4': 210}, 'mask_shape_mismatch': 732, 'caution': 'Blocks and duplicate similarity are proxies, not verified subject identifiers.'}
apo fold 0 gotowy: {"images": 208, "dice_mean": 0.7582725643860719, "dice_median": 0.8634938227912097, "gt_pair_missing": 0, "pred_pair_missing": 0, "wrong_pair": 30, "all_pairs_mt_error_px_n": 208, "all_pairs_mt_error_px_mae": 13.23925984941266, "all_pairs_mt_error_px_p90": 50.509070281139486, "all_pairs_mt_error_px_bias": -0.5745471429248994, "all_pairs_mt_relative_error_n": 208, "all_pairs_mt_relative_error_mae": 0.06046210650268496, "all_pairs_mt_relative_error_p90": 0.3061620190316552, "all_pairs_mt_relative_error_bias": 

In [13]:
# 6. F76 — TRZECIA ARCHITEKTURA PĘCZKÓW, TA SAMA REGUŁA FL.
# Te same pary wejściowe co R22/F71; nowy czysty podział po scaleniu duplikatów.
if TRAIN_F76 and not BUDGET_HIT and len(r76_train.complete_folds(OUT_A))==5:
    paired_f=r76_data.pairs(TRAIN_ROOT/'fasc_images',TRAIN_ROOT/'fasc_masks')
    old_oof=PROJECT/'outputs'/'R22_v1_0_0'/'OOF.csv'
    old=pd.read_csv(old_oof)
    allowed=set(old.file_name.astype(str))
    before=len(paired_f);paired_f=paired_f[paired_f.file_name.isin(allowed)].reset_index(drop=True)
    log('F76: zgodne z R22/F71 pary',len(paired_f),'z',before,'dostępnych par; odrzucone niewykorzystane wcześniej pary wymagają osobnego audytu.')
    frame_f,audit_f=r76_data.audit_and_folds(paired_f,OUT_F,n_folds=5,seed=CFG_F['seed']+1,legacy_folds=old_oof,log=log)
    cache_f=LOCAL/f'r76_fasc_cache_{CFG_F["side"]}_{audit_f["identity"][:10]}'
    r76_data.build_cache(frame_f,cache_f,OUT_F/f'cache_{CFG_F["side"]}.tar','fasc',CFG_F,log)
    pretrained_f=r76_train.prepare_pretrained(PROJECT,RUN,'fasc')
    run_task('fasc',frame_f,cache_f,pretrained_f,CFG_F,FOLDS_F76,OUT_F)
else:log('F76: oczekuje na ukończenie A76 i wolny czas sesji, albo został wyłączony')


F76: zgodne z R22/F71 pary 2046 z 2761 dostępnych par; odrzucone niewykorzystane wcześniej pary wymagają osobnego audytu.
audit: 200/2046
audit: 400/2046
audit: 600/2046
audit: 800/2046
audit: 1000/2046
audit: 1200/2046
audit: 1400/2046
audit: 1600/2046
audit: 1800/2046
audit: 2000/2046
split: {'identity': '2ddc0d53e32588b7ff0008d87f7062dea4807e8942fa47323751f8117859e095', 'samples': 2046, 'groups': 702, 'largest_group': 370, 'exact_or_mirrored_duplicates': 0, 'near_duplicate_links': 307, 'folds': {'0': 409, '1': 409, '2': 409, '3': 410, '4': 409}, 'mask_shape_mismatch': 1344, 'caution': 'Blocks and duplicate similarity are proxies, not verified subject identifiers.'}
fasc fold 0: resumed epoch 9, sample 808/1637
fasc fold 0 epoch 9/20: 908/1637 loss 0.6514; 0.130s/image
fasc fold 0 epoch 9/20: 1008/1637 loss 0.6496; 0.134s/image
fasc fold 0 epoch 9/20: 1108/1637 loss 0.6521; 0.131s/image
fasc fold 0 epoch 9/20: 1208/1637 loss 0.6514; 0.136s/image
fasc fold 0 epoch 9/20: 1308/1637 loss

In [14]:
# 7. OSTATNI EKSPORT I SPIS WAG.
if RUN_EXPORT and not BUDGET_HIT and r76_train.complete_folds(OUT_A):
    try:export_now()
    except r76_train.SessionBudgetReached as e:BUDGET_HIT=True;log(str(e))
weights=[]
for task,folder in [('apo',OUT_A),('fasc',OUT_F)]:
    for done in sorted(folder.glob('fold_*/DONE.json')):
        d=json.loads(done.read_text());p=done.parent/'final.pt'
        weights.append({'task':task,'fold':d['fold'],'relative_path':str(p.relative_to(RUN)),
            'sha256':d['final_sha256'],'bytes':p.stat().st_size,'identity':d['identity']})
r45_core.atomic_json(RUN/'WEIGHTS_MANIFEST.json',{'package_sha256':PACKAGE_SHA,'weights':weights})
status={'a76_folds':r76_train.complete_folds(OUT_A),'f76_folds':r76_train.complete_folds(OUT_F),
        'budget_hit':BUDGET_HIT,'public_score':'NOT_MEASURED','minutes':round((time.time()-T_START)/60,1)}
r45_core.atomic_json(RUN/'RUN_STATUS.json',status)
log('STAN:',json.dumps(status))
latest=RUN/'LATEST_EXPORT.json'
if latest.exists():
    directory=Path(json.loads(latest.read_text())['directory']);log('Ostatni pełny eksport:',directory)
    report=directory/'OSF_COMPARISON.csv'
    if report.exists():
        tab=pd.read_csv(report);show(tab.sort_values(['target','common_mae']),40)
if BUDGET_HIT:log('Bieg zapisany. W nowej sesji: ten sam notebook i RUN_NAME → Uruchom wszystko.')


export already complete: A5_F5_41ceb07de3
STAN: {"a76_folds": [0, 1, 2, 3, 4], "f76_folds": [0, 1, 2, 3, 4], "budget_hit": false, "public_score": "NOT_MEASURED", "minutes": 456.4}
Ostatni pełny eksport: /content/drive/MyDrive/UMUD_Challenge/outputs/R76T_v1_0_0/exports/A5_F5_41ceb07de3
target                         column  n      mae  median_bias  loo_centered_mae  common_n  common_mae  common_centered_mae                  anchor  paired_n  paired_mae  paired_anchor_mae  paired_mae_delta                                                     caution
    fl                 r22_v2_lex5_fl 33 3.652699     2.923890          2.557765        31    3.788209             2.494900     cmb_r22_f71_lex5_fl        32    3.725777           4.261495         -0.535718 Auxiliary external comparison; no leaderboard gain inferred
    fl           r22_a76_safe_lex5_fl 35 3.816041     2.473765          2.772634        31    4.091828             2.737675     cmb_r22_f71_lex5_fl        34    3.897594           

## Co odczytać po treningu i jak podłączyć wynik

| plik | zastosowanie |
|---|---|
| `A76/OOF_SUMMARY.csv` | wybór pary, MT w pikselach, kąt rozcięgna, Dice; nowe foldy po kontroli duplikatów |
| `F76/OOF_SUMMARY.csv` | odzysk pęczków, Dice, błąd kierunku |
| `A76/DATA.json`, `F76/DATA.json` | liczba duplikatów, grup i próbek; ograniczenia podziału |
| `exports/.../test_features.csv` | nowe pomiary do podłączenia do toru R75 |
| `exports/.../osf_features.csv`, `OSF_COMPARISON.csv` | kontrola zewnętrzna, pokrycie i wspólny zbiór obrazów do porównań |
| `*_BASELINE_REPRO.csv` | kontrola R22/F71 względem istniejącego D5; rozbieżność >0,05° wymaga zbadania |
| `*_frags_*_t60.csv.gz`, `*_apo.csv.gz` | linie pęczków i rozcięgien do ponownego liczenia cech offline |
| `WEIGHTS_MANIFEST.json`, `code/`, `ENVIRONMENT.json` | wagi z SHA-256, pełne źródła, wersje bibliotek |

Najpierw porównaj **PA z R22+F71 przy obecnej parze R12B** (R75); to rozdziela już dostępny zysk od zmiany rozcięgien.
Następnie osobno sprawdź MT z `a76_safe` i PA `pa_med6_a76_safe`. `a76_raw` jest kontrolą bez zachowania pary;
`a76_safe` dopasowuje nowe komponenty do dotychczasowej pary i wraca do R12B przy niezgodności.
`a76_v2mix` to dodatkowy kandydat z uśrednieniem map A76/v2 przy zgodnej parze.

Dla FL: `cmb_r22_f71_lex5_fl` jest kontrolą dwóch modeli; `cmb_r22_f71_f76_lex5_fl` dodaje trzeci głos,
z zachowaniem twardego lex5 i progu 0,6. W torze B6 pozostają jego wagi, korekty proporcji i kompensacja zmian na obraz.
Nie należy nakładać surowych kątów pikselowych na już skorygowane kąty fizyczne.

Po potwierdzonej poprawie PA ma sens ponownie dobrać jego shrinkage; wcześniejsze nastawy tłumią także nowy sygnał.
Nie ma podstaw do obiecania konkretnego LB ani awansu na pierwsze miejsce przed pomiarem na serwerze.

**Kolejny ważny etap projektu:** domknąć finalną inferencję od obrazów (skala, modele, dekodery, wszystkie korekty).
Ten notebook nie odtwarza jeszcze kompletnego B6 z obrazów: korzysta z dotychczasowych manifestów skali i eksportuje cechy.
Odtworzenie całego toru jest odrębnym zadaniem koniecznym dla przenoszalności rozwiązania.

## Walidacja przygotowania i źródła

Lokalnie sprawdzono rzeczywiste sieci A76/F76 na małych danych: forward/backward, krótki trening,
reload checkpointów i identyczność końcowych wag po wznowieniu A76 w środku epoki.
Sprawdzono również targety krawędzi na trzech rzeczywistych parach z Drive. To kontrola implementacji;
pełny trening w 1024 px i wynik konkursowy zostaną zmierzone dopiero po uruchomieniu tego pliku na Twoim GPU.

- [Aktualny plan projektu na Drive](https://drive.google.com/file/d/1c__sVFYbAxSe27woR8beaLGEVHf6T6E7/view)
- [Źródła R45T](https://drive.google.com/file/d/1XU6TsbDJUC6jZdjUReXJ8rksKGvK7YSl/view)
- [Źródła R71T v1.1](https://drive.google.com/file/d/1nxwuxEunizXlY2xtxCAC9l-UlW0OZLiK/view)
- [Konkurs: opis i uwaga organizatora o duplikatach](https://www.kaggle.com/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/overview/description)
- [Torchvision ResNet34](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet34.html)
- [PyTorch AMP](https://docs.pytorch.org/docs/stable/amp.html)

Źródła projektu oraz nowy kod: GPL-3.0-or-later. Pełna licencja jest w paczce źródeł.
